In [1]:
# ===== 1. Load the CSV into a SQLite database with 3 tables =====
import pandas as pd
import sqlite3

df = pd.read_csv("Sample - Superstore.csv", encoding="cp1252")
# make column names SQL-friendly: "Order Date" -> order_date
df.columns = [c.lower().replace(" ", "_").replace("-", "_") for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"]).dt.strftime("%Y-%m-%d")
df["ship_date"] = pd.to_datetime(df["ship_date"]).dt.strftime("%Y-%m-%d")

con = sqlite3.connect("superstore.db")

df[["customer_id","customer_name","segment"]].drop_duplicates("customer_id") \
    .to_sql("customers", con, index=False, if_exists="replace")
df[["product_id","product_name","category","sub_category"]].drop_duplicates("product_id") \
    .to_sql("products", con, index=False, if_exists="replace")
df[["row_id","order_id","order_date","ship_date","ship_mode","customer_id",
    "city","state","region","product_id","sales","quantity","discount","profit"]] \
    .to_sql("orders", con, index=False, if_exists="replace")

def sql(query):
    return pd.read_sql(query, con)

print("Database ready: orders, customers, products")

Database ready: orders, customers, products


In [2]:
# ===== Q1. Orders with discount over 50% =====
sql("""
SELECT order_id, region, sales, discount, profit
FROM orders
WHERE discount > 0.5
ORDER BY profit
LIMIT 10
""")

,order_id,region,sales,discount,profit
0,CA-2016-108196,East,4499.985,0.7,-6599.9780
1,CA-2014-169019,Central,2177.584,0.8,-3701.8928
2,CA-2017-134845,West,2549.985,0.7,-3399.9800
3,US-2017-122714,Central,1889.990,0.8,-2929.4845
4,CA-2015-147830,East,1799.994,0.7,-2639.9912
5,CA-2017-131254,Central,1525.188,0.8,-2287.7820
6,CA-2016-130946,Central,1088.792,0.8,-1850.9464
7,CA-2014-165309,Central,896.990,0.8,-1480.0335
8,US-2017-120390,South,1633.188,0.7,-1306.5504
9,CA-2017-128363,South,1614.582,0.7,-1237.8462


In [3]:
# ===== Q2. Sales and profit by region =====
sql("""
SELECT region,
       ROUND(SUM(sales)) AS total_sales,
       ROUND(SUM(profit)) AS total_profit
FROM orders
GROUP BY region
ORDER BY total_profit DESC
""")

,region,total_sales,total_profit
0,West,725458.0,108418.0
1,East,678781.0,91523.0
2,South,391722.0,46749.0
3,Central,501240.0,39706.0


In [4]:
# ===== Q3. Loss-making sub-categories =====
sql("""
SELECT p.sub_category,
       ROUND(SUM(o.profit)) AS total_profit
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.sub_category
HAVING SUM(o.profit) < 0
ORDER BY total_profit
""")

,sub_category,total_profit
0,Tables,-17725.0
1,Bookcases,-3473.0
2,Supplies,-1189.0


In [5]:
# ===== Q4. Margin by category =====
sql("""
SELECT p.category,
       ROUND(SUM(o.sales)) AS sales,
       ROUND(SUM(o.profit)) AS profit,
       ROUND(100 * SUM(o.profit) / SUM(o.sales), 1) AS margin_pct
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY margin_pct
""")


,category,sales,profit,margin_pct
0,Furniture,742000.0,18451.0,2.5
1,Office Supplies,719047.0,122491.0,17.0
2,Technology,836154.0,145455.0,17.4


In [6]:
# ===== Q5. Top 5 loss-making states =====
sql("""
SELECT state, ROUND(SUM(profit)) AS total_profit
FROM orders
GROUP BY state
ORDER BY total_profit
LIMIT 5
""")

,state,total_profit
0,Texas,-25729.0
1,Ohio,-16971.0
2,Pennsylvania,-15560.0
3,Illinois,-12608.0
4,North Carolina,-7491.0


In [7]:
# ===== Q6. Profit by discount band =====
sql("""
SELECT CASE
         WHEN discount = 0    THEN 'No discount'
         WHEN discount <= 0.2 THEN '1-20%'
         WHEN discount <= 0.4 THEN '21-40%'
         ELSE 'Over 40%'
       END AS discount_band,
       COUNT(*) AS orders,
       ROUND(AVG(profit), 1) AS avg_profit
FROM orders
GROUP BY discount_band
ORDER BY avg_profit DESC
""")

,discount_band,orders,avg_profit
0,No discount,4798,66.9
1,1-20%,3803,26.5
2,21-40%,460,-77.9
3,Over 40%,933,-106.7


In [8]:
# ===== Q7. Average shipping days by ship mode =====
sql("""
SELECT ship_mode,
       ROUND(AVG(julianday(ship_date) - julianday(order_date)), 1) AS avg_days
FROM orders
GROUP BY ship_mode
ORDER BY avg_days
""")


,ship_mode,avg_days
0,Same Day,0.0
1,First Class,2.2
2,Second Class,3.2
3,Standard Class,5.0


In [9]:
# ===== Q8. Year-over-year sales growth =====
sql("""
WITH yearly AS (
    SELECT SUBSTR(order_date, 1, 4) AS year, SUM(sales) AS sales
    FROM orders
    GROUP BY year
)
SELECT year,
       ROUND(sales) AS sales,
       ROUND(100.0 * (sales - LAG(sales) OVER (ORDER BY year))
             / LAG(sales) OVER (ORDER BY year), 1) AS growth_pct
FROM yearly
""")

,year,sales,growth_pct
0,2014,484247.0,NaN
1,2015,470533.0,-2.8
2,2016,609206.0,29.5
3,2017,733215.0,20.4


In [10]:
# ===== Q9. Top 3 customers per segment =====
sql("""
SELECT * FROM (
    SELECT c.segment, c.customer_name,
           ROUND(SUM(o.sales)) AS total_sales,
           RANK() OVER (PARTITION BY c.segment ORDER BY SUM(o.sales) DESC) AS rnk
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.segment, c.customer_name
)
WHERE rnk <= 3
""")

,segment,customer_name,total_sales,rnk
0,Consumer,Raymond Buch,15117.0,1
1,Consumer,Adrian Barton,14474.0,2
2,Consumer,Ken Lonsdale,14175.0,3
3,Corporate,Tamara Chand,19052.0,1
4,Corporate,Todd Sumrall,11892.0,2
5,Corporate,Bill Shonely,10502.0,3
6,Home Office,Sean Miller,25043.0,1
7,Home Office,Tom Ashbrook,14596.0,2
8,Home Office,Maria Etezadi,10664.0,3


## SQL Findings
- West is the most profitable region; Central the least.
- Tables, Bookcases and Supplies lose money.
- Texas, Ohio and Pennsylvania have the biggest losses by state.
- Discounts above 20% make the average order unprofitable (1,393 orders).
- Sales fell 2.8% in 2015, then grew 29.5% (2016) and 20.4% (2017).
  